- Breast Cancer Wisconsin (이진분류)
  - 입력(X): 종양의 크기, 둘레, 면적, 질감 등 여러 수치형 특성
  - 타깃(y): 종양 진단 결과(악성 / 양성)

목표
- 동일한 전처리 흐름(결측치 처리 + 표준화)에서 서로 다른 정규화 방식(L2 / L1 / Elastic-Net)을 적용한 로지스틱 회귀 모델을 비교하고,
Accuracy와 F1-score를 기준으로 가장 성능이 좋은 정규화 방식을 자동 선택한다.

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score

In [2]:
# 1. 데이터 로드
# 유방암 진단 데이터셋을 불러옵니다.
# 입력(X)은 여러 수치형 특성이며, 타깃(y)은 악성/양성 여부를 나타냅니다.
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

In [3]:
# 2. 공통 파이프라인 생성 함수
# 결측치 처리 -> 표준화 -> 모델 순서로 구성
# 결측치는 중앙값 기준으로 대체
def make_pipe(model):
    return Pipeline([
        ("imp", SimpleImputer(strategy='median')),   # TODO: 결측치 대체 객체
        ("sc", StandardScaler()),    # TODO: 표준화 객체
        ("mdl", model)
    ])


In [4]:
# 3. 평가 함수
def evaluate(model, X, y, random_state=42):

    # (1) train / test 분리
    # TODO: test_size=0.2 설정
    # TODO: stratify를 사용하여 클래스 비율 유지
    Xtr, Xte, ytr, yte = train_test_split(
        X,
        y,
        test_size=0.2,
        random_state=random_state,
        stratify=y
    )

    # (2) 파이프라인 생성
    # TODO: make_pipe 함수 사용
    pipe = make_pipe(model)

    # (3) 학습
    # TODO: Xtr, ytr 사용하여 학습
    pipe.fit(Xtr, ytr)

    # (4) 예측
    # TODO: Xte를 사용하여 예측
    y_pred = pipe.predict(Xte)

    # (5) 평가
    # TODO: accuracy_score 사용
    # TODO: f1_score 사용
    acc = accuracy_score(yte,y_pred)
    f1 = f1_score(yte, y_pred)

    return {
        "Accuracy": acc,
        "F1": f1
    }


In [5]:
# 4. 모델 정의
# TODO: LogisticRegression 사용
# TODO: solver는 elasticnet도 지원하는 saga로 통일
# TODO: max_iter는 3000
models = [
    (
        "LR_L2",
        LogisticRegression(
            penalty='l2',   # TODO: L2 정규화
            solver="saga",
            max_iter=3000,
            random_state=42
        )
    ),
    (
        "LR_L1",
        LogisticRegression(
            penalty='l1',   # TODO: L1 정규화
            solver="saga",
            max_iter=3000,
            random_state=42
        )
    ),
    (
        "LR_ElasticNet",
        LogisticRegression(
            penalty="elasticnet",
            solver="saga",
            l1_ratio=0.5,
            max_iter=3000,
            random_state=42
        )
    )
]

In [6]:
# 5. 실험 수행
# 각 모델에 대해 evaluate 실행
# 결과에 모델 이름 추가
results = []

for model_name, model in models:
    scores = evaluate(model, X, y, random_state=42)
    scores["Model"] = model_name
    results.append(scores)

c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\SSAFY\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values:

In [7]:
# 6. 결과 정리
# TODO: Accuracy, F1 기준 내림차순 정렬
results_df = pd.DataFrame(results).sort_values(
    ["Accuracy", "F1"], ascending=False
).reset_index(drop=True)

print("=== 정규화 방식 비교 결과 ===")
print(results_df)


=== 정규화 방식 비교 결과 ===
   Accuracy        F1          Model
0  0.991228  0.993103          LR_L1
1  0.982456  0.986111          LR_L2
2  0.982456  0.986111  LR_ElasticNet
